# Amazon CLIP 전체 임베딩만 복구 · 학습 없음
대회 **item_meta.jsonl**의 이미지 URL을 사용하여 기존 v4와 동일한 `clip-ViT-B-32`로 전체 CLIP 임베딩을 다시 추출해. **LightGCN, Transformer, CatBoost는 실행하지 않음.**

Kaggle **GPU T4 + Internet ON**, 대회 `item_meta.jsonl` 데이터셋 Add Input 필요. 이미지 다운로드 및 사전학습 모델 이용이 대회 규칙에 허용되는지 먼저 확인해.

출력: `/kaggle/working/amazon_clip_embeddings.zip`, 내부 `clip_images.npz`와 `clip_images.json`.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util
import torch
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available(): raise RuntimeError('Kaggle Settings에서 GPU 활성화 필요')
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])

In [ ]:
%%writefile /kaggle/working/amazon_clip_only_extract.py
#!/usr/bin/env python3
"""Kaggle-only: Full catalog Amazon image URL -> CLIP embeddings, resumable 256-item checkpoints.
Original JPG files are never stored. Follows the model/encoder from v4.
"""
import os, csv, json, io, hashlib, zipfile, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
from PIL import Image
from sentence_transformers import SentenceTransformer
import torch

ROOT=Path('/kaggle/input')
OUT=Path('/kaggle/working/clip_recovery')
OUT.mkdir(parents=True, exist_ok=True)
NPZ=OUT/'clip_images.npz'
STATE=OUT/'clip_images.json'
ZIP=Path('/kaggle/working/amazon_clip_embeddings.zip')
MAX_IMAGES=int(os.environ.get('MAX_IMAGES','0'))

def meta_path():
    candidates=sorted(ROOT.rglob('item_meta.jsonl'))
    if not candidates:
        raise FileNotFoundError('Kaggle Add Input에서 대회 item_meta.jsonl 연결')
    for f in candidates:
        if 'ssu-data-science-amazon-top-k-recommendation' in str(f):return f
    return candidates[0]

def image_url(r):
    for key in ('image_url','imageURL','img_url','image','imageUrl','url'):
        v=r.get(key)
        if isinstance(v,str) and v.startswith(('http://','https://')):return v
    v=r.get('images')
    if isinstance(v,str):
        try:v=json.loads(v)
        except Exception:return v if v.startswith('http') else ''
    if isinstance(v,dict):v=[v]
    if isinstance(v,list):
        for x in v:
            if isinstance(x,str) and x.startswith('http'):return x
            if isinstance(x,dict):
                for k in ('hi_res','large','url','thumb'):
                    s=x.get(k)
                    if isinstance(s,str) and s.startswith('http'):return s
    return ''

def main():
    meta=meta_path()
    urls={}
    with meta.open(encoding='utf-8') as f:
        for line in f:
            try:d=json.loads(line)
            except Exception:continue
            ident=d.get('item_id');u=image_url(d)
            if ident is not None and u:urls[str(ident)]=u
    if len(urls)<1000:raise RuntimeError(f'Expected many image URLs; found {len(urls)}. Check item_meta mapping.')
    entries=sorted(urls.items())
    if MAX_IMAGES>0 and len(entries)>MAX_IMAGES:
        select=np.sort(np.random.default_rng(42).choice(len(entries),size=MAX_IMAGES,replace=False))
        entries=[entries[int(i)] for i in select]
    fingerprint=hashlib.sha256(json.dumps(entries).encode()).hexdigest()
    keys=[];vectors=[];resume=0
    if NPZ.exists() and STATE.exists():
        try:
            status=json.loads(STATE.read_text())
            if status.get('stamp')==fingerprint:
                resume=min(int(status.get('next_index',0)),len(entries))
                with np.load(NPZ,allow_pickle=False) as d:
                    keys=list(map(str,d['item_ids']));vectors=list(d['vectors'])
                print('✅ Resume partial CLIP:',resume,'/',len(entries),'cached:',len(keys),flush=True)
        except Exception as e:
            print('Old checkpoint invalid; restarting:',repr(e),flush=True)
            resume=0;keys=[];vectors=[]
    if resume<len(entries):
        model=SentenceTransformer('clip-ViT-B-32', device='cuda' if torch.cuda.is_available() else 'cpu')
        def fetch(entry):
            it,url=entry
            try:
                r=requests.get(url,timeout=8,headers={'User-Agent':'Mozilla/5.0'})
                r.raise_for_status()
                if len(r.content)>8_000_000:return None
                img=Image.open(io.BytesIO(r.content)).convert('RGB')
                img.thumbnail((384,384))
                return it,img
            except Exception:return None
        print('CLIP URLs:',len(entries),'resume_from:',resume,flush=True)
        with ThreadPoolExecutor(max_workers=8) as pool:
            for pos in range(resume,len(entries),256):
                samples=list(pool.map(fetch,entries[pos:pos+256]))
                for offset in range(0,len(samples),32):
                    batch=[x for x in samples[offset:offset+32] if x is not None]
                    if not batch:continue
                    enc=model.encode([im for _,im in batch],batch_size=16,
                           convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False)
                    for (item,_),v in zip(batch,enc):
                        keys.append(item);vectors.append(v)
                if vectors:
                    np.savez_compressed(NPZ,item_ids=np.asarray(keys),vectors=np.asarray(vectors,dtype=np.float32))
                STATE.write_text(json.dumps({'stamp':fingerprint,'next_index':pos+len(samples),
                      'success_images':len(keys),'total':len(entries)},indent=2))
                print('CLIP progress',pos+len(samples),'/',len(entries),'success=',len(keys),flush=True)
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    if not NPZ.exists():raise RuntimeError('CLIP download had no successes')
    with np.load(NPZ,allow_pickle=False) as d:
        n=len(d['item_ids']); shape=d['vectors'].shape
    with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_STORED) as z:
        z.write(NPZ,arcname='clip_images.npz')
        z.write(STATE,arcname='clip_images.json')
    print('✅ CLIP ready:',n,'vectors',shape,flush=True)
    print('✅ Download NPZ:',NPZ,flush=True)
    print('✅ Download ZIP:',ZIP,flush=True)
    print('ℹ️ Download or Save Version now, before closing the session.',flush=True)
if __name__=='__main__':main()


In [ ]:
import os,subprocess,sys
MAX_IMAGES = 0 # 전체 19,543 URL. 일부 테스트만 할 때 300
os.environ['MAX_IMAGES']=str(MAX_IMAGES)
subprocess.run([sys.executable,'-u','/kaggle/working/amazon_clip_only_extract.py'],check=True)

In [ ]:
from pathlib import Path
from IPython.display import FileLink
p=Path('/kaggle/working/amazon_clip_embeddings.zip')
assert p.exists(), '아직 CLIP ZIP이 생성되지 않았어'
print('저장:',p,'size MB:',round(p.stat().st_size/1048576,2))
display(FileLink(str(p)))

완료하면 **즉시 ZIP을 다운로드하거나 Save Version으로 Output을 보존**해. 이후 비공개 Kaggle Dataset으로 생성해서 제출 노트북에서 Add Input으로 연결해. 원본 JPG는 포함되지 않아.